# UN System Data Commons — the companion notebook

Runs everything shown in videos **11 (the REST API)** and **12 (point an AI assistant at it)**,
against the live platform, so you can see the figures move rather than take the video's word for it.

**No API key. No account. No install** — the whole notebook uses the Python standard library.

| Section | What it proves |
|---|---|
| 1 · One series | The API returns the same 115.11 the website shows |
| 2 · Facets | A value and its meaning arrive in *separate* objects |
| 3 · A whole region | One expression instead of thirty-seven requests |
| 4 · The `LATEST` trap | That one response mixes eleven reference years |
| 5 · MCP | An assistant fetching the figure instead of recalling it |
| 6 · A defensible export | A CSV where every row carries its own provenance |

Last verified against the live platform on 18 September 2026.

In [ ]:
import json
import urllib.parse
import urllib.request

API = "https://unsd-datacommons.gcp.un-icc.cloud"
MCP = f"{API}/mcp"


def post(path, body):
    """POST JSON, get JSON back."""
    req = urllib.request.Request(
        API + path,
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)


def get(path, params):
    """GET with repeated query keys, which the graph endpoint relies on."""
    qs = urllib.parse.urlencode(params, doseq=True)
    with urllib.request.urlopen(f"{API}{path}?{qs}", timeout=60) as r:
        return json.load(r)


print("ready — no key, no account")

## 1 · One series, for places you already know

The two identifiers below are the only thing you need. Both are visible in the platform's own
download dialog — see video 06.

- **Variable** `undata/sdg/SH_STA_MORT.SEX--F` — maternal mortality ratio, female
- **Entity** `country/BGD` — Bangladesh, ISO 3166 alpha-3

In [ ]:
VARIABLE = "undata/sdg/SH_STA_MORT.SEX--F"
PLACE = "country/BGD"

res = post("/api/observations/series", {"variables": [VARIABLE], "entities": [PLACE]})
block = res["data"][VARIABLE][PLACE]

for point in block["series"][-5:]:
    print(f"{point['date']}   {point['value']:>10.5f}")

print(f"\n{len(block['series'])} observations in total")

That last figure is the one the website puts in its headline tile, and the one video 01 reads off
the screen by hand.

## 2 · Facets: where the meaning lives

This is the step people skip. The numbers come back in `data`; what they *mean* comes back in
`facets`, keyed by an id on each series. Join them, or you will publish a bare number with no unit.

In [ ]:
facet = res["facets"][str(block["facet"])]

for key in ("provenanceUrl", "unitDisplayName", "observationPeriod", "importName"):
    if key in facet:
        print(f"{key:<20} {facet[key]}")

latest = block["series"][-1]
print(
    f"\nA sentence you can defend:\n"
    f"  Bangladesh's maternal mortality ratio was {latest['value']:.1f} "
    f"{facet.get('unitDisplayName', '')} in {latest['date']}.\n"
    f"  Source: {facet.get('provenanceUrl', 'unknown')}"
)

## 3 · A whole region, without listing it

The single highest-leverage thing in the API. `entity.expression` walks the graph rather than
enumerating country codes:

```
africa<-containedInPlace+{typeOf:Country}
```

Read it as: *everything contained in Africa, recursively, that is a Country.* Swap `africa` for
any of the 30 UN regions — `SouthernAsia`, `Melanesia`, `LatinAmericaAndCaribbean`.

In [ ]:
HOMICIDE = "undata/sdg/VC_IHR_PSRC"
REGION = "africa"

region = get(
    "/core/api/v2/observation",
    {
        "date": "LATEST",
        "variable.dcids": HOMICIDE,
        "entity.expression": f"{REGION}<-containedInPlace+{{typeOf:Country}}",
        "select": ["date", "value", "entity", "variable"],
    },
)

by_entity = region["byVariable"][HOMICIDE]["byEntity"]
rows = []
for entity, payload in by_entity.items():
    ob = payload["orderedFacets"][0]["observations"][0]
    rows.append((entity, ob["date"], ob["value"]))

print(f"{len(rows)} countries returned from one request")

## 4 · The `LATEST` trap

`date=LATEST` resolves **per country**, not across the set. So a single response is not a
snapshot — it is a collage of whatever year each country last reported.

This is video 04's whole argument, and here it is falling out of one API call.

In [ ]:
years = sorted({date for _, date, _ in rows})
print(f"reference years present: {years[0]} -> {years[-1]}  ({len(years)} distinct)\n")

newest = sorted(rows, key=lambda r: r[1])[-1]
oldest = sorted(rows, key=lambda r: r[1])[0]
print(f"newest   {newest[0]:<14} {newest[1]}   {newest[2]}")
print(f"oldest   {oldest[0]:<14} {oldest[1]}   {oldest[2]}")
print(
    f"\nThese two bars would sit side by side in a ranking, "
    f"{int(newest[1]) - int(oldest[1])} years apart, with nothing on the chart saying so."
)

### Pinning a year instead

The honest alternative. Note the trade: countries that did not report that year drop out entirely,
rather than falling back to an older reading. Choose it deliberately, and say which you chose.

In [ ]:
PINNED = "2020"

pinned = get(
    "/core/api/v2/observation",
    {
        "date": PINNED,
        "variable.dcids": HOMICIDE,
        "entity.expression": f"{REGION}<-containedInPlace+{{typeOf:Country}}",
        "select": ["date", "value", "entity", "variable"],
    },
)
pinned_rows = [
    e for e, p in pinned["byVariable"][HOMICIDE]["byEntity"].items()
    if p.get("orderedFacets")
]

print(f"LATEST      {len(rows):>3} countries, {len(years)} different years")
print(f"date={PINNED}   {len(pinned_rows):>3} countries, 1 year")
print(f"\nComparability costs you {len(rows) - len(pinned_rows)} countries. That is the real trade-off.")

## 5 · MCP — letting an assistant fetch it

The same platform speaks the **Model Context Protocol** over streamable HTTP, which is what
video 12 connects to. You would normally point a client at it rather than calling it by hand —
in Claude Code that is one line:

```bash
claude mcp add --transport http undata https://unsd-datacommons.gcp.un-icc.cloud/mcp
```

But calling it directly shows what your assistant is actually doing on your behalf.

In [ ]:
def mcp(method, params=None, rid=1):
    """One JSON-RPC call. The server answers as SSE, so pull the data: lines out."""
    req = urllib.request.Request(
        MCP,
        data=json.dumps({"jsonrpc": "2.0", "id": rid, "method": method,
                         "params": params or {}}).encode(),
        headers={"Content-Type": "application/json",
                 "Accept": "application/json, text/event-stream"},
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        for line in r.read().decode().splitlines():
            if line.startswith("data: "):
                return json.loads(line[6:])
    return {}


info = mcp("initialize", {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "undc-notebook", "version": "1.0"},
})["result"]["serverInfo"]
print(f"{info['name']}  v{info['version']}\n")

for tool in mcp("tools/list", rid=2)["result"]["tools"]:
    print(f"  {tool['name']}")

### The two-step an assistant actually performs

Plain words in, identifier out, then observation. Nothing is recalled from a model's memory.

In [ ]:
def call(name, args, rid):
    out = mcp("tools/call", {"name": name, "arguments": args}, rid)
    return json.loads(out["result"]["content"][0]["text"])


found = call("search_indicators",
             {"query": "maternal mortality", "places": ["Bangladesh"], "per_search_limit": 3},
             rid=3)

print("candidate indicators:")
for dcid, name in found["dcidNameMappings"].items():
    if dcid.startswith("undata/") and "." in dcid:
        print(f"  {dcid}\n      {name}")

obs = call("get_observations",
           {"variable_dcid": VARIABLE, "place_dcid": PLACE, "date_range_start": "2020"},
           rid=4)

print("\nobservations:")
for place, date, value in obs["data"]["rows"]:
    print(f"  {place}  {date}  {value}")
print(f"\nsource: {obs['sourceMetadata']['provenanceUrl']}")

Same figure as section 1, reached from plain English rather than from an identifier you had to
know. That is the difference MCP makes — and the unit and provenance still arrive attached.

## 6 · An export that stays defensible

The failure mode is a spreadsheet that gets forwarded until nobody can say where the numbers came
from. Put the identifier and the source on **every row**, not in a header someone will delete.

In [ ]:
import csv
import datetime

OUT = "maternal-mortality-bangladesh.csv"
retrieved = datetime.date.today().isoformat()

with open(OUT, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["entity", "variable_dcid", "year", "value", "unit", "source", "retrieved"])
    for point in block["series"]:
        w.writerow([
            PLACE, VARIABLE, point["date"], point["value"],
            facet.get("unitDisplayName", ""), facet.get("provenanceUrl", ""), retrieved,
        ])

print(f"wrote {OUT}\n")
print(open(OUT).read().split("\n")[0])
print(open(OUT).read().split("\n")[-2])

## Where to go next

- **Change `PLACE`** to any ISO alpha-3 country, as `country/KEN`, `country/IND`.
- **Change `REGION`** to `SouthernAsia`, `WesternAfrica`, `Melanesia` — the containment expression
  does not care how many countries come back.
- **Find other variables** with `search_indicators` in section 5, or by browsing the
  thematic catalogue (video 09).

Three things worth carrying out of this notebook:

1. A value without its facet is a number without a unit.
2. `LATEST` is a per-country answer, not a date.
3. An identifier is permanent; a search phrase is not. Record the identifier in your method note.